# ClauseNexa — Legal-BERT 7-class training (Colab)

Retrains `nlpaueb/legal-bert-base-uncased` with a 7th class, **Other**, so out-of-scope clauses
(Notices, Entire Agreement, Assignment, Force Majeure, ...) are no longer forced into one of the 6 categories.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Training takes roughly 45–90 minutes on a T4.

Settings: grouped 70/15/15 split (0 contract overlap) · `max_length=384` · 3 epochs · batch 8 · lr 2e-5 ·
weight decay 0.01 · fp16 · best epoch chosen by **Macro F1**.

In [4]:
# STEP 0 — Install libraries (Colab already has torch, pandas, scikit-learn)
!pip install -q transformers datasets accelerate

## Step 1 — Settings and data file

In [5]:
import os

# The dataset built by ml/scripts/05_build_7class_dataset.py
DATA_FILE = "ClauseNexa_LEDGAR_7Class.csv"

# Where the trained model is saved (Google Drive, so it survives the Colab session)
DRIVE_DIR = "/content/drive/MyDrive/ClauseNexa"
OUTPUT_DIR = f"{DRIVE_DIR}/models/legalbert_classifier_7class"
CHECKPOINT_DIR = "/content/checkpoints"   # temporary, not saved to Drive

SMOKE_TEST = False   # True = tiny run to check the code works (not a real model)

BASE_MODEL = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 384
RANDOM_STATE = 42

In [6]:
# Mount Google Drive and find the dataset.
# Put ClauseNexa_LEDGAR_7Class.csv in  My Drive/ClauseNexa/  first,
# OR this cell asks you to upload it from your laptop
# (C:\Users\Kavyah\Desktop\capstone_1\ClauseNexa\data\processed\).
from google.colab import drive, files

drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)

DATA_PATH = f"{DRIVE_DIR}/{DATA_FILE}"

if not os.path.exists(DATA_PATH):
    print("Dataset not found in Drive — please upload", DATA_FILE)
    uploaded = files.upload()
    DATA_PATH = f"/content/{list(uploaded)[0]}"

print("Using dataset:", DATA_PATH)

Mounted at /content/drive
Dataset not found in Drive — please upload ClauseNexa_LEDGAR_7Class.csv


Saving ClauseNexa_LEDGAR_7Class.csv to ClauseNexa_LEDGAR_7Class.csv
Using dataset: /content/ClauseNexa_LEDGAR_7Class.csv


## Step 2 — Load data and encode the 7 labels

In [7]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

# Label ids in alphabetical order (same convention as the 6-class model)
CATEGORIES = sorted(df["clause_category"].unique())
ID2LABEL = dict(enumerate(CATEGORIES))
LABEL2ID = {label: i for i, label in ID2LABEL.items()}

df["label_id"] = df["clause_category"].map(LABEL2ID)

assert len(CATEGORIES) == 7, CATEGORIES
print("Dataset shape:", df.shape)
print("Labels:", ID2LABEL)
print(df["clause_category"].value_counts())

Dataset shape: (51359, 6)
Labels: {0: 'Confidentiality', 1: 'Governing Law', 2: 'Liability', 3: 'Non-Compete', 4: 'Other', 5: 'Payment', 6: 'Termination'}
clause_category
Governing Law      20512
Other               9000
Termination         7223
Payment             6033
Confidentiality     5481
Liability           1722
Non-Compete         1388
Name: count, dtype: int64


## Step 3 — Grouped 70/15/15 split (no contract in more than one split)

In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss_1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss_1.split(df, groups=df["source"]))
train_df, temp_df = df.iloc[train_idx].copy(), df.iloc[temp_idx].copy()

gss_2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_idx, test_idx = next(gss_2.split(temp_df, groups=temp_df["source"]))
val_df, test_df = temp_df.iloc[val_idx].copy(), temp_df.iloc[test_idx].copy()

train_s, val_s, test_s = set(train_df["source"]), set(val_df["source"]), set(test_df["source"])
print("Train / Validation / Test:", len(train_df), len(val_df), len(test_df))
print("Contract overlap  train-val:", len(train_s & val_s),
      "| train-test:", len(train_s & test_s), "| val-test:", len(val_s & test_s))

assert not (train_s & val_s) and not (train_s & test_s) and not (val_s & test_s), "Contract leakage!"

print(pd.DataFrame({
    "train": train_df["clause_category"].value_counts(),
    "val": val_df["clause_category"].value_counts(),
    "test": test_df["clause_category"].value_counts(),
}))

if SMOKE_TEST:
    train_df = train_df.sample(64, random_state=RANDOM_STATE)
    val_df = val_df.sample(32, random_state=RANDOM_STATE)
    test_df = test_df.sample(32, random_state=RANDOM_STATE)

Train / Validation / Test: 35993 7628 7738
Contract overlap  train-val: 0 | train-test: 0 | val-test: 0
                 train   val  test
clause_category                   
Governing Law    14274  3130  3108
Other             6360  1316  1324
Termination       5077  1041  1105
Payment           4273   864   896
Confidentiality   3838   810   833
Liability         1209   250   263
Non-Compete        962   217   209


## Step 4 — Tokenize (max_length = 384)

In [9]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_dataset(frame):
    ds = Dataset.from_pandas(frame[["clause_text", "label_id"]], preserve_index=False)
    # Truncate at 384 tokens; padding is done per batch by the data collator (faster)
    ds = ds.map(lambda b: tokenizer(b["clause_text"], truncation=True, max_length=MAX_LENGTH),
                batched=True)
    return ds.rename_column("label_id", "labels").remove_columns(["clause_text"])

train_ds, val_ds, test_ds = to_dataset(train_df), to_dataset(val_df), to_dataset(test_df)
collator = DataCollatorWithPadding(tokenizer)
print(train_ds)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/35993 [00:00<?, ? examples/s]

Map:   0%|          | 0/7628 [00:00<?, ? examples/s]

Map:   0%|          | 0/7738 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 35993
})


## Step 5 — Model and training settings (best epoch by Macro F1)

In [10]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(CATEGORIES), id2label=ID2LABEL, label2id=LABEL2ID,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=1 if SMOKE_TEST else 3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    seed=RANDOM_STATE,
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collator, compute_metrics=compute_metrics)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to T4 GPU!")

[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

GPU: Tesla T4


## Step 6 — Train (~45–90 min on T4)

In [11]:
trainer.train()

# Validation Macro F1 for every epoch
for log in trainer.state.log_history:
    if "eval_macro_f1" in log:
        print(f"Epoch {log['epoch']:.0f}: val Macro F1 = {log['eval_macro_f1']:.6f}  "
              f"accuracy = {log['eval_accuracy']:.6f}")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.137338,0.100485,0.979287,0.970948,0.979336
2,0.049880,0.085015,0.985055,0.978940,0.985079
3,0.017589,0.084021,0.985711,0.978431,0.985702


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1: val Macro F1 = 0.970948  accuracy = 0.979287
Epoch 2: val Macro F1 = 0.978940  accuracy = 0.985055
Epoch 3: val Macro F1 = 0.978431  accuracy = 0.985711


## Step 7 — Final evaluation on the untouched test set (all 7 classes)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

output = trainer.predict(test_ds)
y_true = output.label_ids
y_pred = np.argmax(output.predictions, axis=-1)

print("TEST RESULTS")
print("Accuracy:   ", round(accuracy_score(y_true, y_pred), 6))
print("Macro F1:   ", round(f1_score(y_true, y_pred, average="macro"), 6))
print("Weighted F1:", round(f1_score(y_true, y_pred, average="weighted"), 6))
print()
print(classification_report(y_true, y_pred, labels=list(range(len(CATEGORIES))),
                            target_names=CATEGORIES, digits=4, zero_division=0))

In [ ]:
# Confusion matrix (rows = true class, columns = predicted class)
cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=list(range(len(CATEGORIES)))),
                  index=[f"true {c}" for c in CATEGORIES],
                  columns=[f"pred {c}" for c in CATEGORIES])
print(cm.to_string())

# The confusions that matter most for the Other class
other = LABEL2ID["Other"]
print("\nKey Other confusions (test set):")
for name in ["Payment", "Governing Law", "Termination"]:
    k = LABEL2ID[name]
    print(f"  true Other -> predicted {name:14s}: {int(((y_true == other) & (y_pred == k)).sum())}")
    print(f"  true {name:13s} -> predicted Other   : {int(((y_true == k) & (y_pred == other)).sum())}")

                      pred Confidentiality  pred Governing Law  pred Liability  pred Non-Compete  pred Other  pred Payment  pred Termination
true Confidentiality                   826                   0               0                 4           1             0                 2
true Governing Law                       0                3101               0                 0           7             0                 0
true Liability                           0                   1             254                 0           6             1                 1
true Non-Compete                         9                   0               0               197           2             1                 0
true Other                               2                   6               1                 0        1303             0                12
true Payment                             0                   0               1                 0           5           880                10
true Terminat

In [ ]:
# Which kinds of Other clauses are recognised (accuracy per Other group, test set)
test_eval = test_df.copy()
test_eval["predicted"] = [ID2LABEL[i] for i in y_pred]
other_test = test_eval[test_eval["clause_category"] == "Other"]

def most_common_wrong(predicted):
    wrong = predicted[predicted != "Other"]
    return wrong.mode().iat[0] if len(wrong) else "-"

per_group = (
    other_test.assign(correct=other_test["predicted"] == "Other")
    .groupby("other_group")
    .agg(clauses=("correct", "size"), accuracy=("correct", "mean"),
         most_common_wrong=("predicted", most_common_wrong))
    .sort_values("accuracy")
)
print(per_group.round(3).to_string())

## Step 8 — Specific test clauses (4 out-of-scope + 1 per original category)

In [ ]:
TEST_CLAUSES = [
    ("Other (Notices)", "All notices under this Agreement must be in writing and delivered by hand, courier or email to the addresses set out in the Schedule, and are deemed received on the next business day."),
    ("Other (Entire Agreement)", "This Agreement constitutes the entire agreement between the parties and supersedes all prior negotiations, representations and understandings, whether written or oral."),
    ("Other (Assignment)", "Neither party may assign or transfer any of its rights or obligations under this Agreement without the prior written consent of the other party."),
    ("Other (Force Majeure)", "Neither party is liable for any delay or failure to perform caused by events beyond its reasonable control, including fire, flood, pandemic, war or government action."),
    ("Confidentiality", "The Receiving Party shall keep all Confidential Information strictly confidential and shall not disclose it to any third party without the Disclosing Party's prior written consent."),
    ("Governing Law", "This Agreement is governed by the laws of New South Wales, Australia, and the parties submit to the non-exclusive jurisdiction of its courts."),
    ("Liability", "The Supplier's total aggregate liability under this Agreement shall not exceed the fees paid in the twelve months preceding the claim, and neither party is liable for indirect or consequential loss."),
    ("Non-Compete", "The Employee shall not, for twelve months after the termination of employment, engage in any business that competes with the Company within the Sydney metropolitan area."),
    ("Payment", "The Customer shall pay each invoice within thirty days of receipt by electronic funds transfer to the account nominated by the Supplier."),
    ("Termination", "Either party may terminate this Agreement by giving the other party thirty days' written notice, or immediately if the other party commits a material breach."),
]

model.eval()
device = next(model.parameters()).device
correct = 0

print(f"{'expected':26s} {'predicted':16s} {'confidence':>10s}")
for expected, text in TEST_CLAUSES:
    inputs = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0]
    label = ID2LABEL[int(probs.argmax())]
    ok = expected.startswith(label)
    correct += ok
    print(f"{expected:26s} {label:16s} {float(probs.max()):10.4f}  {'OK' if ok else 'WRONG'}")

print(f"\n{correct}/{len(TEST_CLAUSES)} correct")

## Step 9 — Save the model to Google Drive

In [ ]:
import json

if SMOKE_TEST:
    print("Smoke test — model not saved.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    trainer.save_model(OUTPUT_DIR)          # model.safetensors + config.json (with the 7 labels)
    tokenizer.save_pretrained(OUTPUT_DIR)   # tokenizer files
    with open(f"{OUTPUT_DIR}/label_mapping.json", "w") as f:
        json.dump({str(i): l for i, l in ID2LABEL.items()}, f, indent=4)
    print("Saved to:", OUTPUT_DIR)
    print(os.listdir(OUTPUT_DIR))